# Prepare Dynamic Few-Shot Data (run once, before the dynamic notebooks)

Builds the two files the `notebook_<model>_dynamic` notebooks need and saves them to Drive `NLP Project/Data/`:

1. **`example_pool.jsonl`** — 500 candidate demonstration examples streamed from `facebook/natural_reasoning` on HuggingFace (no local dataset needed). Each has a reference answer (demo answer) and a Llama-3.3-70B response (CoT reasoning demo). The 100 test questions are excluded.
2. **`dynamic_examples.json`** — for each test question, the top-3 most similar pool examples by sentence-embedding cosine similarity (all-MiniLM-L6-v2).

**Runtime**: CPU is enough (no GPU needed). Takes ~10–20 min total, most of it streaming the dataset.

**Needs**: `sampled.jsonl` on Drive in `NLP Project/Data/` (already there), and your HF token in Colab secrets (the dataset is public but authenticated access avoids rate limits).

In [1]:
!pip install -q datasets sentence-transformers

In [3]:
import json
import random
import re
from collections import Counter

import numpy as np
from datasets import load_dataset
from google.colab import drive, userdata
from huggingface_hub import login

In [4]:
try:
    login(token=userdata.get("HF_TOKEN"))
    print("Logged into HuggingFace")
except Exception as e:
    print(f"No HF token ({e}) — continuing anonymously, the dataset is public")

drive.mount('/content/drive')

Logged into HuggingFace
Mounted at /content/drive


## Configuration & test set

In [5]:
DRIVE_DATA_PATH = "/content/drive/MyDrive/NLP Project/Data"
SEED = 42

# Test set (minus no_answer) is 10/37/35 single_word/short/long -> scaled to 500
TARGETS = {"single_word": 60, "short": 220, "long": 220}

# How many dataset rows to stream. 300k eligible-checked rows gives thousands
# of candidates per bucket -- plenty for a 500-example pool.
SCAN_LIMIT = 300_000

MAX_QUESTION_WORDS = 150
MAX_ANSWER_WORDS = 80
MAX_REASONING_WORDS = 150   # ~200 tokens
MIN_REASONING_WORDS = 30

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
K = 3

with open(f"{DRIVE_DATA_PATH}/sampled.jsonl", encoding="utf-8") as f:
    test_set = [json.loads(line) for line in f]
test_questions = {item["question"].strip() for item in test_set}
print(f"Loaded {len(test_set)} test questions (will be excluded from the pool)")

Loaded 100 test questions (will be excluded from the pool)


In [6]:
def classify_answer(ref_answer):
    if not ref_answer or ref_answer.strip() == "":
        return "no_answer"
    words = ref_answer.strip().split()
    if len(words) == 1:
        return "single_word"
    elif len(words) <= 20:
        return "short"
    return "long"


def clean_reasoning(text):
    """Strip markdown step headers and the trailing 'final answer' line
    (the demo answer is appended separately as 'ANSWER: ...')."""
    text = re.sub(r"^#+\s*", "", text, flags=re.MULTILINE)
    idx = text.lower().rfind("the final answer is")
    if idx > len(text) * 0.5:
        text = text[:idx]
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def truncate_at_sentence(text, max_words):
    words = text.split()
    if len(words) <= max_words:
        return text
    truncated = " ".join(words[:max_words])
    ends = list(re.finditer(r"[.!?](?=\s|$)", truncated))
    if ends:
        return truncated[: ends[-1].end()]
    return truncated + "."


def eligible(row):
    ref = (row.get("reference_answer") or "").strip()
    if classify_answer(ref) == "no_answer":
        return False
    if len(ref.split()) > MAX_ANSWER_WORDS:
        return False
    q = (row.get("question") or "").strip()
    if not q or len(q.split()) > MAX_QUESTION_WORDS:
        return False
    if q in test_questions:
        return False
    responses = row.get("responses") or []
    if not responses or not (responses[0].get("response") or "").strip():
        return False
    reasoning = clean_reasoning(responses[0]["response"])
    if len(reasoning.split()) < MIN_REASONING_WORDS:
        return False
    return True

## 1. Build the example pool (streamed from HuggingFace)

In [7]:
# Stream the dataset and reservoir-sample each bucket to its target size.
# Reservoir sampling = every eligible row in the scanned range has an equal
# chance of ending up in the pool, without holding the dataset in memory.
rng = random.Random(SEED)
reservoirs = {cat: [] for cat in TARGETS}
seen = {cat: 0 for cat in TARGETS}

ds = load_dataset("facebook/natural_reasoning", split="train", streaming=True)

scanned = 0
for row in ds:
    scanned += 1
    if scanned > SCAN_LIMIT:
        break
    if scanned % 25_000 == 0:
        sizes = {c: len(r) for c, r in reservoirs.items()}
        print(f"  scanned {scanned:,} rows... reservoirs: {sizes}")
    if not eligible(row):
        continue
    cat = classify_answer(row["reference_answer"])
    seen[cat] += 1
    slim = {"question": row["question"].strip(),
            "reference_answer": row["reference_answer"].strip(),
            "response": row["responses"][0]["response"]}
    target = TARGETS[cat]
    if len(reservoirs[cat]) < target:
        reservoirs[cat].append(slim)
    else:
        j = rng.randrange(seen[cat])
        if j < target:
            reservoirs[cat][j] = slim

print(f"\nScanned {scanned - 1:,} rows. Eligible seen per bucket: {seen}")
for cat, res in reservoirs.items():
    assert len(res) == TARGETS[cat], f"Bucket {cat}: only {len(res)}/{TARGETS[cat]} — raise SCAN_LIMIT"
print("All buckets filled.")

README.md:   0%|          | 0.00/2.42k [00:00<?, ?B/s]

  scanned 25,000 rows... reservoirs: {'single_word': 60, 'short': 220, 'long': 220}
  scanned 50,000 rows... reservoirs: {'single_word': 60, 'short': 220, 'long': 220}
  scanned 75,000 rows... reservoirs: {'single_word': 60, 'short': 220, 'long': 220}
  scanned 100,000 rows... reservoirs: {'single_word': 60, 'short': 220, 'long': 220}
  scanned 125,000 rows... reservoirs: {'single_word': 60, 'short': 220, 'long': 220}
  scanned 150,000 rows... reservoirs: {'single_word': 60, 'short': 220, 'long': 220}
  scanned 175,000 rows... reservoirs: {'single_word': 60, 'short': 220, 'long': 220}
  scanned 200,000 rows... reservoirs: {'single_word': 60, 'short': 220, 'long': 220}
  scanned 225,000 rows... reservoirs: {'single_word': 60, 'short': 220, 'long': 220}
  scanned 250,000 rows... reservoirs: {'single_word': 60, 'short': 220, 'long': 220}
  scanned 275,000 rows... reservoirs: {'single_word': 60, 'short': 220, 'long': 220}
  scanned 300,000 rows... reservoirs: {'single_word': 60, 'short': 2

In [8]:
records = []
for cat, res in reservoirs.items():
    for row in res:
        records.append({
            "question": row["question"],
            "answer": row["reference_answer"],
            "reasoning": truncate_at_sentence(clean_reasoning(row["response"]), MAX_REASONING_WORDS),
            "answer_type": cat,
        })

rng.shuffle(records)
for idx, rec in enumerate(records):
    rec["pool_id"] = idx

POOL_PATH = f"{DRIVE_DATA_PATH}/example_pool.jsonl"
with open(POOL_PATH, "w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print(f"Saved {len(records)} examples to {POOL_PATH}")

# Leakage check
overlap = {r["question"] for r in records} & test_questions
assert not overlap, f"LEAKAGE: {len(overlap)} pool questions are in the test set!"
print("Leakage check passed: no overlap with the 100 test questions")
print("Pool answer types:", Counter(r["answer_type"] for r in records))

Saved 500 examples to /content/drive/MyDrive/NLP Project/Data/example_pool.jsonl
Leakage check passed: no overlap with the 100 test questions
Pool answer types: Counter({'long': 220, 'short': 220, 'single_word': 60})


## 2. Select top-3 similar examples per test question

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBED_MODEL)
pool_emb = model.encode([r["question"] for r in records],
                        normalize_embeddings=True, show_progress_bar=True)
test_emb = model.encode([t["question"] for t in test_set],
                        normalize_embeddings=True, show_progress_bar=True)

sims = test_emb @ pool_emb.T  # (100, 500) cosine similarities

selection = {}
for row_i, item in enumerate(test_set):
    top_idx = np.argsort(sims[row_i])[-K:]  # ascending: least -> most similar
    selection[str(item["sample_id"])] = [
        {
            "pool_id": records[j]["pool_id"],
            "question": records[j]["question"],
            "answer": records[j]["answer"],
            "reasoning": records[j]["reasoning"],
            "similarity": round(float(sims[row_i][j]), 4),
        }
        for j in top_idx
    ]
print(f"Selected top-{K} examples for {len(selection)} test questions")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Selected top-3 examples for 100 test questions


In [10]:
OUT_PATH = f"{DRIVE_DATA_PATH}/dynamic_examples.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(selection, f, ensure_ascii=False, indent=2)
print(f"Saved to {OUT_PATH}")

all_sims = [ex["similarity"] for exs in selection.values() for ex in exs]
print(f"\nSimilarity: mean={np.mean(all_sims):.3f}, "
      f"min={np.min(all_sims):.3f}, max={np.max(all_sims):.3f}")

usage = Counter(ex["pool_id"] for exs in selection.values() for ex in exs)
print(f"Distinct pool examples used: {len(usage)} / {K * len(selection)} slots")
print(f"Most reused: {usage.most_common(3)}")

# Prompt-size proxy check (words; tokens are roughly 1.3x): worst-case prompt
# must stay well under the 3072-token budget used by the dynamic notebooks.
worst = max(
    sum(len(ex['question'].split()) + len(ex['reasoning'].split()) + len(ex['answer'].split())
        for ex in exs) + len(item['question'].split())
    for item, exs in zip(test_set, (selection[str(it['sample_id'])] for it in test_set))
)
print(f"Worst-case prompt size: ~{worst} words (~{int(worst * 1.3)} tokens) — budget is 3072")

Saved to /content/drive/MyDrive/NLP Project/Data/dynamic_examples.json

Similarity: mean=0.363, min=0.158, max=0.680
Distinct pool examples used: 224 / 300 slots
Most reused: [(461, 4), (15, 3), (4, 3)]
Worst-case prompt size: ~823 words (~1069 tokens) — budget is 3072


In [11]:
for item in test_set[:3]:
    exs = selection[str(item["sample_id"])]
    print(f"TEST Q (sample_id={item['sample_id']}, type={item['answer_type']}):")
    print(f"  {item['question'][:150]}")
    for ex in exs:
        print(f"    [{ex['similarity']:.3f}] {ex['question'][:120]}")
    print()

TEST Q (sample_id=0, type=short):
  Given the information about Company B and Company D, including their contribution margins and fixed costs, calculate the breakeven point for each comp
    [0.315] A hole is punched at a height h in the side of a container of height H, filled with water. Using Bernoulli's Equation an
    [0.329] Given two objects with movement equations \(x_1(t) = 2t + 1\) and \(y_1(t) = 4t^2\) for the first object, and \(x_2(t) =
    [0.368] Consider the Boxcar Burger Restaurant problem, where the restaurant has $2.7 million available for expansion and current

TEST Q (sample_id=1, type=no_answer):
  Prove or disprove that an upper bound for $\rho(u)$ necessarily implies that $x/\Gamma(a+1)$ is an upper bound for $\Psi(x, y)$, where $\Psi(x, y) \ap
    [0.273] Consider the Euclidean path integral for QCD with massive fermions charged under QCD. Show that the QCD vacuum energy E(
    [0.314] Derive the expression for the critical density as a function of the scale fac

## Done

Both files are now on Drive in `NLP Project/Data/`. Sanity-check the preview above — retrieved examples should look topically related to their test questions.

Next: run `notebook_llama_dynamic.ipynb`, `notebook_mistral_dynamic.ipynb`, `notebook_gemma_dynamic.ipynb` (one Colab session each, T4 GPU), then `notebook_comparison.ipynb`.

Optionally download `example_pool.jsonl` and `dynamic_examples.json` into the repo's `Data/` folder and commit them for reproducibility.